# Validate JetOps Raw Event Capture

This notebook supports two execution modes:

- Databricks: reads the raw container with `dbutils` and Spark.
- Local VS Code notebook: enumerates captured Avro blobs with Azure CLI and decodes them with `fastavro`.

The defaults target the Terraform-managed dev environment. Override them with `JETOPS_*` environment variables when you need to point at a different storage account, Event Hub namespace, or Event Hub name.

In [5]:
import os

storage_account = os.getenv("JETOPS_STORAGE_ACCOUNT", "stherbalifedev001")
container = os.getenv("JETOPS_RAW_CONTAINER", "raw")
secret_scope = os.getenv("JETOPS_SECRET_SCOPE", "herbalife-storage")
secret_key = os.getenv("JETOPS_SECRET_KEY", "raw-sas-token")
eventhub_namespace = os.getenv("JETOPS_EVENTHUB_NAMESPACE", "evh-herbalife-dev")
eventhub_name = os.getenv("JETOPS_EVENTHUB_NAME", "jetops-maintenance-events-dev")
capture_root = os.getenv("JETOPS_CAPTURE_ROOT", "jetops-maintenance")
resource_group = os.getenv("JETOPS_RESOURCE_GROUP", "rg-herbalife-dev-core")
az_cli = os.getenv("AZURE_CLI_PATH", r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd")

capture_path = (
    f"wasbs://{container}@{storage_account}.blob.core.windows.net/"
    f"{capture_root}/{eventhub_namespace}/{eventhub_name}/*/*/*/*/*/*/*"
)

is_databricks = "dbutils" in globals() and "spark" in globals()
print(f"Execution mode: {'databricks' if is_databricks else 'local'}")
print(f"Capture path: {capture_path}")

if is_databricks:
    sas_token = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    spark.conf.set(
        f"fs.azure.sas.{container}.{storage_account}.blob.core.windows.net",
        sas_token,
    )
else:
    print("Local mode will enumerate Avro blobs under the capture prefix with Azure CLI.")

Execution mode: local
Capture path: wasbs://raw@stherbalifedev001.blob.core.windows.net/jetops-maintenance/evh-herbalife-dev/jetops-maintenance-events-dev/*/*/*/*/*/*/*
Local mode will enumerate Avro blobs under the capture prefix with Azure CLI.


In [7]:
import json
import subprocess
import tempfile
from pathlib import Path

maintenance_schema = """
event_type STRING,
event_id STRING,
event_time_utc STRING,
tail_number STRING,
aircraft_model STRING,
maintenance_log_id INT,
work_order_id STRING,
status STRING,
maintenance_type STRING,
component STRING,
fault_code STRING,
severity STRING,
part_hours DOUBLE,
inspection_date STRING,
technician_id STRING,
hangar STRING,
airport_code STRING,
details STRING,
ingestion_source STRING,
schema_version STRING
"""

if is_databricks:
    from pyspark.sql.functions import col, decode, from_json

    capture_df = spark.read.format("avro").load(capture_path)
    parsed_df = (
        capture_df
        .select(from_json(decode(col("Body"), "UTF-8"), maintenance_schema).alias("event"))
        .select("event.*")
    )

    display(parsed_df.orderBy(col("event_time_utc").desc()))
else:
    try:
        from fastavro import reader
    except ImportError as exc:
        raise ImportError(
            "Local mode requires fastavro in the notebook kernel. Install it before running this cell."
        ) from exc

    capture_prefix = f"{capture_root}/{eventhub_namespace}/{eventhub_name}"
    account_key = subprocess.check_output(
        [
            az_cli,
            "storage",
            "account",
            "keys",
            "list",
            "--resource-group",
            resource_group,
            "--account-name",
            storage_account,
            "--query",
            "[0].value",
            "-o",
            "tsv",
        ],
        text=True,
    ).strip()

    file_list = subprocess.check_output(
        [
            az_cli,
            "storage",
            "fs",
            "file",
            "list",
            "--account-name",
            storage_account,
            "--account-key",
            account_key,
            "--file-system",
            container,
            "--path",
            capture_prefix,
            "--exclude-dir",
            "-o",
            "json",
        ],
        text=True,
    )
    files = json.loads(file_list)
    avro_files = sorted(file_info["name"] for file_info in files if file_info["name"].endswith(".avro"))
    if not avro_files:
        raise FileNotFoundError(f"No Avro capture files found under {capture_prefix}")

    latest_file = avro_files[-1]
    with tempfile.TemporaryDirectory() as temp_dir:
        local_file = Path(temp_dir) / Path(latest_file).name
        subprocess.run(
            [
                az_cli,
                "storage",
                "fs",
                "file",
                "download",
                "--account-name",
                storage_account,
                "--account-key",
                account_key,
                "--file-system",
                container,
                "--path",
                latest_file,
                "--destination",
                str(local_file),
                "--overwrite",
                "true",
            ],
            check=True,
            capture_output=True,
            text=True,
        )

        with local_file.open("rb") as handle:
            records = list(reader(handle))

    payloads = [json.loads(record["Body"].decode("utf-8")) for record in records]
    print(f"Latest file: {latest_file}")
    print(f"Decoded records: {len(payloads)}")
    payloads[:5]

Latest file: jetops-maintenance/evh-herbalife-dev/jetops-maintenance-events-dev/1/2026/04/05/03/49/50.avro
Decoded records: 12618
